# 🤖 Social Media Post AI Agent
### Build Story: From Architecture to Working Demo

---

## 🎯 Objective

Build an AI agent that creates consistent, on-brand social media posts for any company or product. The agent remembers past work — so on return visits, the user skips straight to generating new posts in the same style without re-researching anything.

**The problem this solves:** Creating social media content manually is time-consuming, inconsistent across sessions, and hard to scale. A simple ChatGPT prompt isn't enough because it has no memory of past brand guidelines, no ability to search for company info, and no way to run multiple platform-specific agents simultaneously.

---

## 🧠 Agentic Pattern: Sequential + Parallel Workflow

This project combines two patterns from class:

1. **Sequential Workflow** (like `orchestrator.py` from class) — Research → Style Guide → Image Selection → Post Generation → Scheduling. Each stage depends on the previous one.

2. **Parallel Multi-Agent** (like `07_workflow_multitasking.ipynb` from class) — Within post generation, one specialized agent per platform runs simultaneously using `ThreadPoolExecutor`. An Instagram Specialist, Twitter Specialist, and Blog Post Writer all work at the same time.

3. **ReAct + Function Calling** (upgraded from `02_react_agent.ipynb` from class) — The research agent reasons through Thought/Action/Observation steps and calls tools (web search, URL fetch, document read, ask user) via OpenAI function calling instead of regex parsing.

---

## 📁 Folder Structure Created Automatically

```
AI Storage/
  {Company}/
    {Company} Company Report.json
    Products/{Product} Product Report.json
    Style Guides/{Product} Style Guide.json
    Images/{Product}/          ← grows over time
    Created Posts/{date} – Request {Product}/
      Post 1/caption.txt + post_image.png
    Log Files/
```

---

**Setup:** Fill in `.env` with your `OPENAI_API_KEY` before running. See `README.md` for full setup instructions.

---
## 📦 Step 0: Imports & Config

Imports all local agent modules. Each module is a separate Python file in the same folder — this keeps the code organized and mirrors the class project structure where each agent lives in its own file (e.g. `twitter_agent.py`, `parallel_workflow.py`).

In [ ]:
import os, json, re, datetime, time
from pathlib import Path

from agent_storage  import Storage
from agent_research import ResearchAgent
from agent_posts    import PLATFORM_AGENTS, build_context, parse_platforms
from agent_parallel import run_all_posts
from agent_schedule import ScheduleAgent
from agent_logger   import Logger
from agent_utils    import confirm, print_receipt

print('✅ Imports OK')

---
## 🔑 Step 1: API Keys

Uses `python-dotenv` to load keys from a `.env` file — the same pattern as the class `openai_client.py` which calls `load_dotenv()` at the top of the file. Keys are never hardcoded in the notebook itself, which is important for the GitHub repo (`.env` is in `.gitignore`).

`OPENAI_REVIEW_KEY` uses the same key but feeds into the cheaper `gpt-3.5-turbo` model used by the A/B reviewer — keeping costs down while still scoring post quality.

In [ ]:
from dotenv import load_dotenv

# Loads all keys from .env automatically — same pattern as class openai_client.py
load_dotenv()

OPENAI_API_KEY        = os.getenv('OPENAI_API_KEY')
OPENAI_REVIEW_KEY     = os.getenv('OPENAI_API_KEY')   # same key, cheaper model used internally
GOOGLE_CALENDAR_ID    = os.getenv('GCAL_ID', 'primary')
GCAL_CREDENTIALS_JSON = os.getenv('GCAL_CREDENTIALS', 'credentials.json')

if not OPENAI_API_KEY:
    raise ValueError('OPENAI_API_KEY not found. Did you fill in your .env file?')

print('✅ Keys loaded from .env')

---
## 📁 Step 2: Storage Paths

`Storage` is the file system manager for the whole project. It handles all folder creation, JSON reading/writing, and image management. Centralizing file I/O in one class means no agent ever writes files directly — they all go through `Storage`. This makes it easy to change the folder structure in one place without touching any agent code.

In [ ]:
PROJECT_ROOT = Path('.')   # same folder as this notebook
storage = Storage(PROJECT_ROOT / 'AI Storage')
print(f'📂 AI Storage: {storage.root.resolve()}')

---
## 🗣️ Step 3: Parse User Request

**Prompt Engineering — Request Parsing**

The user types a natural language request and `parse_request()` extracts structured fields from it using regex patterns. This is the first prompt engineering decision in the project: rather than asking the user to fill out a form, they can write naturally and the system figures out the intent.

The parsed output is called a **receipt** — it's the order that drives the rest of the workflow.

**Image mode parsing** is also handled here. Platform defaults are defined in `IMAGE_PLATFORM_DEFAULTS` in `agent_utils.py` — easy to find and modify:
- Instagram → `Provided Images` by default
- Twitter / Blog → `No` by default
- `"with image"` on any platform → `Provided Images`
- `"AI generated"` explicitly → `AI Generated`
- `"without image"` → `No`

**Try changing `USER_REQUEST` below** to see how the parser handles different inputs.

In [ ]:
# ── Change this to your request ────────────────────────────────────────────
USER_REQUEST = "Create 1 Instagram and 2 Twitter posts for Rita's Kiwi Melon in June + Schedule. Make it an interactive post."

from agent_utils import parse_request
receipt = parse_request(USER_REQUEST)
print_receipt(receipt)

---
## 🔍 Step 4: Company & Product Lookup

**Agentic Pattern: ReAct + Function Calling**

This is where the research agent runs. It uses a combination of two patterns from class:

**From `02_react_agent.ipynb`:** The ReAct prompt (`REACT_PROMPT` in `agent_research.py`) tells GPT to reason step by step — Thought → Action → PAUSE → Observation → Answer. You can watch the thinking happen live in the output below.

**Upgraded from class:** The class version used regex (`ACTION_RE`) to parse action names out of raw GPT text. This project upgrades to **OpenAI function calling** — GPT signals which tool it wants via a structured `tool_calls` object instead of plain text. This is more reliable because GPT can't accidentally misspell the action name.

**Four tools available to the agent:**
| Tool | When used |
|---|---|
| `web_search` | Company/product has an online presence |
| `fetch_url` | User provided a specific URL |
| `read_document` | User uploaded a file (PDF, txt) |
| `ask` | Nothing found — agent pauses and asks the user |

**On return visits:** If the company and product reports already exist locally, the agent loads them instantly and skips all research — the user just hits Enter.

In [ ]:
researcher = ResearchAgent(
    storage=storage,
    openai_key=OPENAI_API_KEY,
)

# Optional: set these if you have a specific URL or uploaded document
# provided_url  = 'https://example.com/about'
# uploaded_file = 'my_product_brief.pdf'
provided_url  = None
uploaded_file = None


In [ ]:
company_report, product_report, resolved_company, resolved_product = researcher.resolve(
    company_name=receipt['company'],
    product_name=receipt['product'],
    provided_url=provided_url,
    uploaded_file=uploaded_file,
)

# Update receipt with the canonical names resolved by the agent
# (e.g. 'Rita\'s' becomes 'Rita\'s Water Ice' after fuzzy match / official name lookup)
receipt['company'] = resolved_company
receipt['product'] = resolved_product
print(f'  ✅ Receipt updated: company=[{resolved_company}], product=[{resolved_product}]')

---
## 📖 Step 5: Review Reports (Optional)

Gives the user a chance to inspect the generated Company and Product reports before continuing. Since these are saved as JSON files in `AI Storage/`, the user can also edit them directly in any text editor and press Enter to continue with the updated version.

**On return visits:** This step is quick — the reports are already accurate from the first run, so the user typically just hits Enter to skip.

In [ ]:
want_review = confirm('Would you like to review the Company or Product report before continuing? (y/N)')
if want_review:
    choice = input("Type 'company', 'product', or 'both': ").strip().lower()
    if choice in ('company', 'both'):
        print(json.dumps(company_report, indent=2))
    if choice in ('product', 'both'):
        print(json.dumps(product_report, indent=2))
    input('\nPress Enter when done reviewing...')

---
## ✅ Step 6: Confirm Receipt

Shows the parsed receipt and lets the user modify any field before generation begins.
This is the last chance to change platforms, post count, image mode, etc.

**How to modify:** Type `modify <field> to <value>` — for example `modify num_posts to 5`.
Press Enter or type `all good` to confirm.

In [ ]:
from agent_utils import interactive_receipt_editor
receipt = interactive_receipt_editor(receipt)
print_receipt(receipt)

---
## 🎨 Step 7: Style References

Add screenshots of posts you want the agent to imitate — any platform, any image mode.
These are passed to GPT Vision during caption drafting so the agent sees the style,
tone, and humor of real posts. They also inform the style guide generated in Step 7.5.

References live in `AI Storage/{Company}/Images/{Product}/references/` and persist
across sessions — add them once and they're used every time.

**Commands:** `addref <filepath>` | `rmref <number>` | Enter to continue

In [ ]:
ref_dir = storage.references_dir(receipt['company'], receipt['product'])
ref_dir.mkdir(parents=True, exist_ok=True)

# Snapshot BEFORE the loop so we can detect any changes after
refs_before_loop = set(r.name for r in storage.list_references(receipt['company'], receipt['product']))
existing_refs = storage.list_references(receipt['company'], receipt['product'])

print(f'\n🎨 Style references for [{receipt["product"]}]:')
if existing_refs:
    print(f'  Found {len(existing_refs)} reference(s):')
    for i, r in enumerate(existing_refs, 1):
        print(f'    {i}. {r.name}')
else:
    print('  No style references yet.')

print('\n  Commands: addref <filepath> | rmref <number> | Enter')
print('  Tip: pasting a file path directly also works as addref.')
print()

while True:
    cmd = input('  ref> ').strip().strip('"').strip("'")
    if not cmd:
        break

    parts  = cmd.split(None, 1)
    action = parts[0].lower()
    arg    = parts[1].strip() if len(parts) > 1 else ''

    # Auto-detect file path — treat as addref without typing the command
    is_path = ('\\' in cmd or '/' in cmd or
               cmd.lower().endswith(('.png', '.jpg', '.jpeg')))
    if is_path and action not in ('addref', 'rmref'):
        action = 'addref'
        arg    = cmd

    if action == 'addref' and arg:
        added = storage.add_reference_to_library(receipt['company'], receipt['product'], arg)
        if added:
            existing_refs = storage.list_references(receipt['company'], receipt['product'])
    elif action == 'rmref' and arg.isdigit():
        idx = int(arg) - 1
        if 0 <= idx < len(existing_refs):
            existing_refs[idx].unlink()
            existing_refs = storage.list_references(receipt['company'], receipt['product'])
            print('  🗑️  Removed.')
        else:
            print('  ❌ Invalid number.')
    else:
        print('  addref <filepath> | rmref <number> | Enter')

reference_images = storage.list_references(receipt['company'], receipt['product'])
print(f'\n✅ {len(reference_images)} style reference(s) loaded.')

In [ ]:
# ── Detect reference changes and ask about style guide update ────────────
# Compares references before and after Step 7.
# If changed and a style guide already exists, offers 3 options.
# Future improvement: replace numbered menu with GPT classification
# (same sentiment_analysis pattern from class) — see prompt engineering notes.

refs_after   = set(r.name for r in storage.list_references(receipt['company'], receipt['product']))
refs_changed = refs_after != refs_before_loop  # compare against snapshot taken before Step 7
has_existing_guide = storage.style_guide_exists(receipt['company'], receipt['product'])

update_mode = 'leave'  # default

if refs_changed and has_existing_guide:
    print('\n🎨 References changed and a style guide already exists.')
    print('  What would you like to do?')
    print('  1. Leave as-is (references still influence posts directly)')
    print('  2. Touch up   (lightly blend new references into existing style)')
    print('  3. Regenerate (full rewrite using all current references)')
    choice = input('  Choice (1/2/3, default 1): ').strip()
    if choice == '2':
        update_mode = 'touchup'
    elif choice == '3':
        update_mode = 'regenerate'
    else:
        update_mode = 'leave'
    print(f'  ✅ Style guide update mode: {update_mode}')
else:
    print('ℹ️  No reference changes detected — style guide will load as-is if it exists.')

---
## 🎨 Step 7.5: Generate Style Guide

**Prompt Engineering — Style Consistency**

The style guide captures brand vibe, tone, color palette, emoji usage, and visual themes.
It is now generated AFTER style references are added (Step 7) so GPT Vision can
analyze the reference screenshots and factor their style directly into the guide.

**On return visits:** Loads existing style guide instantly — no questions asked.
Use the File Manager (`demo.py`) to update it if needed.

**Using a base:** If this is a new product but other style guides exist, you can use
one as a starting point. Reference screenshots can optionally be copied across too.

In [ ]:
style_guide = researcher.resolve_style_guide(
    company_report=company_report,
    product_report=product_report,
    reference_images=reference_images,
    update_mode=update_mode,
)
style_vibe = style_guide.get('vibe', '')
print(f'\n✅ Style guide ready. Vibe: {style_vibe}')


---
## 🖼️ Step 7.6: Image Selection

Only runs if `images` in the receipt is not `No`.
Manages the per-product photo library for actual post images (enhance or DALL-E).
Separate from style references — these are the real photos that appear in the post.

**Commands:** `add <filepath>` | `select <number>` | `remove <number>` | Enter to continue

In [ ]:
image_mode             = receipt.get('images', 'No')
selected_images        = []
enhance_as_inspiration = False  # asked once here, passed to agents via base_brief

if image_mode == 'No':
    print('⏭️  Image mode is No — skipping image selection.')
else:
    company = receipt['company']
    product = receipt['product']
    img_dir = storage.images_dir(company, product)
    img_dir.mkdir(parents=True, exist_ok=True)
    existing = storage.list_images(company, product)

    print(f'\n🖼️  Image library for [{product}]:')
    if existing:
        print(f'  Found {len(existing)} product photo(s):')
        for i, p in enumerate(existing, 1):
            print(f'    {i}. {p.name}')
    else:
        print('  No product photos in library yet.')

    print('\n  Commands: add <filepath> | select <number> | all | remove <number> | Enter')
    print('  Tip: pasting a file path directly also works as add.')
    print()

    while True:
        cmd = input('  > ').strip().strip('"').strip("'")
        if not cmd:
            break

        parts  = cmd.split(None, 1)
        action = parts[0].lower()
        arg    = parts[1].strip() if len(parts) > 1 else ''

        # Auto-detect file path
        is_path = ('\\' in cmd or '/' in cmd or
                   cmd.lower().endswith(('.png', '.jpg', '.jpeg')))
        if is_path and action not in ('add', 'select', 'remove', 'all'):
            action = 'add'
            arg    = cmd

        if action in ('all', 'select all') or (action == 'select' and arg.lower() == 'all'):
            for img in existing:
                if img not in selected_images:
                    selected_images.append(img)
            print(f'  ✅ Selected all {len(existing)} image(s).')
            break  # no need to hit Enter — continue automatically
        elif action == 'add' and arg:
            added = storage.add_image_to_library(company, product, arg)
            if added:
                existing = storage.list_images(company, product)
                if added not in selected_images:
                    selected_images.append(added)
                    print(f'  ✅ Added and selected: {added.name}')
        elif action == 'select' and arg.isdigit():
            idx = int(arg) - 1
            if 0 <= idx < len(existing):
                chosen = existing[idx]
                if chosen not in selected_images:
                    selected_images.append(chosen)
                    print(f'  ✅ Selected: {chosen.name}')
                else:
                    print(f'  Already selected: {chosen.name}')
            else:
                print(f'  ❌ Invalid number.')
        elif action == 'remove' and arg.isdigit():
            idx = int(arg) - 1
            if 0 <= idx < len(existing):
                target = existing[idx]
                target.unlink()
                existing = storage.list_images(company, product)
                selected_images = [p for p in selected_images if p != target]
                print(f'  🗑️  Removed: {target.name}')
            else:
                print(f'  ❌ Invalid number.')
        else:
            print('  add <filepath> | select <number> | all | remove <number> | Enter')

    if image_mode == 'Provided Images' and not selected_images:
        if not existing:
            print('\n  ℹ️  No images — falling back to AI Generated.')
            image_mode = 'AI Generated'
            receipt['images'] = 'AI Generated'

    print(f'\n✅ Image mode: {image_mode}')
    print(f'   Selected: {[p.name for p in selected_images] or "none"}')

---
## ✍️ Step 8: Generate Posts (Parallel Workflow)

**Agentic Pattern: Parallel Multi-Agent Execution**

This step directly mirrors `07_workflow_multitasking.ipynb` from class. The same three components:

1. **Specialized agents** — `InstagramAgent`, `TwitterAgent`, `BlogAgent` each inherit from `BaseAgent` (same as `TwitterAgent` inheriting from `BaseAgent` in class). Each has one `execute()` method.

2. **`ParallelWorkflow`** — uses `ThreadPoolExecutor` to run all agents simultaneously (identical to class `ParallelWorkflow`). Multiple posts across multiple platforms all generate at the same time.

3. **A/B scoring loop** — each agent drafts a post, a cheaper reviewer model scores it (0-10), and the loop keeps the best version up to `max_tries` attempts. This is in `_ab_loop()` in `agent_base.py`.

**Prompt Engineering — Caption + Image Cohesion:**
Within each agent's `execute()`, the caption is generated first via the A/B loop. The final caption is then passed as context to the image generation step — so DALL-E or the image edit endpoint knows what story the post is telling. This was a deliberate design choice: image_context alone produces generic images, but caption + image_context produces images that match the post's message.

**New vs class:** `base_brief` now carries `image_mode`, `selected_images`, and `style_vibe` alongside context — these flow through to every agent automatically via the `{**base_brief, 'post_num': n}` spread.

In [ ]:
# ── 1. Build shared context from reports ───────────────────────────────
context = build_context(
    company=company_report,
    product=product_report,
    style=style_guide,
    extra=receipt.get('additional_info', ''),
    month=receipt.get('when', ''),
)

# ── 2. Build agents — all platforms run in parallel within each round ────────
# e.g. '1 Instagram + 1 Twitter' → 1 round: [Instagram, Twitter] together
# e.g. '2 Instagram + 1 Twitter' → 2 rounds: [Insta,Twitter], [Insta]
platforms       = parse_platforms(receipt.get('platforms', 'instagram'))
n_posts         = int(receipt.get('num_posts', 1))
valid_platforms = [p for p in platforms if p in PLATFORM_AGENTS]
n_rounds        = max(1, n_posts // len(valid_platforms)) if valid_platforms else n_posts

post_platform_list = []
agents_per_post    = []

for round_idx in range(n_rounds):
    round_agents = [
        PLATFORM_AGENTS[p](openai_key=OPENAI_API_KEY, review_key=OPENAI_REVIEW_KEY)
        for p in valid_platforms
    ]
    agents_per_post.append(round_agents)
    post_platform_list.extend(valid_platforms)

# Handle remainder
remainder = n_posts - (n_rounds * len(valid_platforms))
for p in valid_platforms[:remainder]:
    agents_per_post.append([PLATFORM_AGENTS[p](openai_key=OPENAI_API_KEY, review_key=OPENAI_REVIEW_KEY)])
    post_platform_list.append(p)

print(f'🚀 Starting post generation...\n' + '=' * 70)
print(f'📋 Platforms : {", ".join(p.capitalize() for p in post_platform_list)}')
print(f'   Posts     : {len(post_platform_list)} total')
print(f'   Images    : {image_mode}')
print('=' * 70 + '\n')

# ── 3. Build base brief ──────────────────────────────────────────────
base_brief = {
    'context':                context,
    'total_posts':            len(post_platform_list),
    'image_mode':             image_mode,
    'selected_images':        selected_images,
    'reference_images':       reference_images,
    'style_vibe':             style_vibe,
    'logo_description':       company_report.get('logo_description', '') or '',
    'additional_info':        receipt.get('additional_info', ''),
    'enhance_as_inspiration': enhance_as_inspiration,
    'brand_context': ' | '.join(filter(None, [
        f"Notes: {receipt.get('additional_info','')}" if receipt.get('additional_info') else '',
        f"Vibe: {style_guide.get('vibe','')}" if style_guide.get('vibe') else '',
        f"Tone: {style_guide.get('tone','')}" if style_guide.get('tone') else '',
        f"Product: {product_report.get('product_name','')} — {str(product_report.get('product_description',''))[:80]}" if product_report.get('product_name') else '',
        f"Company: {company_report.get('company_name','')}" if company_report.get('company_name') else '',
    ])),
}


In [ ]:
start_time = time.time()
posts      = run_all_posts(agents_per_post, base_brief)
elapsed    = time.time() - start_time

print(f'\n✨ All agents finished in {elapsed:.2f} seconds')
print('=' * 70)

# ── A/B Score Summary ────────────────────────────────────────────────────
print('\n📊 A/B SCORE SUMMARY')
print('-' * 40)
for i, post in enumerate(posts, 1):
    platform = post.get('platform','?').upper()
    score    = post.get('score', 'N/A')
    audit    = post.get('audit_result', {})
    passed   = audit.get('passed', True)
    reason   = audit.get('reason', '')
    score_str = f'{score:.1f}/10' if isinstance(score, float) else str(score)
    audit_str = '✅ Image passed audit' if passed else f'❌ Image audit failed: {reason}'
    has_image = post.get('image_bytes') is not None
    print(f'  Post {i} [{platform}]  Score: {score_str}  '
          f'{audit_str if has_image else "(no image)"}')
print('-' * 40)

# ── Post content display ─────────────────────────────────────────────────
print('\n📊 GENERATED SOCIAL MEDIA CONTENT')
print('=' * 70 + '\n')

for i, post in enumerate(posts, 1):
    has_image = post.get('image_bytes') is not None
    score     = post.get('score', 'N/A')
    score_str = f'{score:.1f}/10' if isinstance(score, float) else str(score)
    label     = f'Post {i} [{post.get("platform","?").upper()}] — Score: {score_str} {"🖼️" if has_image else ""}'
    print(f'┌─ {label} ' + '─' * max(0, 68 - len(label)))
    print('│')
    for line in post.get('caption', '').split('\n'):
        print(f'│ {line}')
    print('└' + '─' * 69)
    print()

print('=' * 70)
print(f'✅ Generation complete! Total time: {elapsed:.2f}s')
print('=' * 70)


---
## 💾 Step 9: Save Posts & Images

`save_posts()` creates the output folder structure and writes `caption.txt` for each post. Images are saved separately via `save_image()` which writes the raw PNG bytes returned by DALL-E or the image edit endpoint.

The receipt is also saved as `Receipt of Creation.txt` alongside the posts so there's always a record of what was requested.

In [ ]:
# Save captions and receipt to the output folder
output_dir = storage.save_posts(
    company=receipt['company'],
    product=receipt['product'],
    posts=posts,
    receipt=receipt,
)

# Save any generated or enhanced images into their post subfolder
# image_bytes is None if image_mode was No or generation failed
for i, post in enumerate(posts, 1):
    if post.get('image_bytes'):
        post_dir = output_dir / f'Post {i}'
        storage.save_image(post_dir, post['image_bytes'], filename='post_image.png')

print(f'\n📁 All posts saved to: {output_dir}')

---
## 📅 Step 10: Schedule Posts (Optional)

If `schedule` in the receipt is `Yes`, this step:
1. Asks GPT to suggest ideal posting dates for the given month and platform
2. Shows the suggestions and asks the user to confirm (up to 3 attempts)
3. Pushes confirmed dates to Google Calendar via the Google Calendar API

**Setup required:** Google Calendar credentials JSON from Google Cloud Console. See `README.md` for step-by-step instructions. Skipped automatically if `schedule` is `No`.

In [ ]:
if receipt.get('schedule') == 'Yes':
    scheduler = ScheduleAgent(
        openai_key=OPENAI_API_KEY,
        credentials_json=GCAL_CREDENTIALS_JSON,
        calendar_id=GOOGLE_CALENDAR_ID,
    )
    scheduler.run(
        receipt=receipt,
        posts=posts,
        output_dir=output_dir,
    )
else:
    print('⏭️  Scheduling skipped.')

---
## 📝 Step 11: Write Log

Writes a structured audit log to `AI Storage/{Company}/Log Files/`. The log records the full receipt, a summary of the company and product reports, the style guide vibe, every generated caption with its A/B score, and the output folder path.

This mirrors the audit trail pattern from `agency_core.py` in class — every run is traceable.

In [ ]:
logger   = Logger(storage)
log_path = logger.write(
    receipt=receipt,
    company_report=company_report,
    product_report=product_report,
    style_guide=style_guide,
    posts=posts,
    output_dir=output_dir,
)
print(f'\n🎉 All done! Log: {log_path}')

---
## 🗂️ File Manager — View & Update Core Files

In addition to creating posts, the agent can view and update the three core files:
**Company Report**, **Product Report**, and **Style Guide**.
Logs are view-only.

**How intent detection works:**

Before every message, a small GPT call runs `INTENT_PROMPT` to classify what the user wants:
- `create_posts` → normal post creation flow
- `manage` → file manager side flow
- `quit` → exit

This is the same classification pattern as `sentiment_analysis` from `prompts_text_usecases.py`
in class — natural language in, structured label out.
Extended here to also extract company, product, and file_type so the agent knows exactly what to act on.

**Guardrail:** Only `company_report`, `product_report`, and `style_guide` can be edited.
Any attempt to edit a log is blocked and redirected to view-only.

**Four options when managing a file:**
1. **Auto-update** — re-runs the full research loop and rewrites the file
2. **Specific change** — user describes change in plain English, GPT applies it, loops until confirmed
3. **View only** — no changes made
4. **Exit** — go back to main loop

The cell below shows an example of loading and displaying a file directly from the notebook:

In [ ]:
# Example: view a company report directly in the notebook
# Uses the same fuzzy match as the research agent —
# so 'Rita\'s' finds 'Rita\'s Water Ice', 'Rita\'s Watch Ice' finds 'Rita\'s Water Ice' etc.
from difflib import SequenceMatcher

company_input = "Rita's Water Ice"   # ← change this
product_input = "Kiwi Melon"         # ← change this

# Fuzzy match company
existing_companies = storage.list_companies()
input_lower = company_input.lower()
company_match = next(
    (c for c in existing_companies if
     SequenceMatcher(None, input_lower, c.lower()).ratio() > 0.6
     or input_lower in c.lower() or c.lower() in input_lower),
    None
)

if not company_match:
    print(f"No company found matching '{company_input}'. Run Step 4 first.")
else:
    if company_match != company_input:
        print(f"Matched '{company_input}' → [{company_match}]")
    company = company_match

    # Fuzzy match product
    products_dir = storage.products_dir(company)
    existing_products = [
        f.stem.replace(' Product Report', '')
        for f in products_dir.glob('*.json')
    ] if products_dir.exists() else []

    p_lower = product_input.lower()
    product_match = next(
        (p for p in existing_products if
         SequenceMatcher(None, p_lower, p.lower()).ratio() > 0.6
         or p_lower in p.lower() or p.lower() in p_lower),
        product_input
    )
    product = product_match

    # Load and display
    report = storage.load_company_report(company)
    print(f"Company: {report.get('company_name')}")
    print(f"Industry: {report.get('industry')}")
    print(f"Brand voice: {report.get('brand_voice')}")
    print()
    print(json.dumps(report, indent=2))


---
## 🚀 Live Demo

The cells above document how the agent was built step by step.

**To run the full interactive agent** (post creation + file management):

```bash
python demo.py
```

The demo uses three additional class patterns:
- **`simpleChatBot_chatgpt.py`** — full conversation history maintained
- **`generate_text_stream()` from `openai_client.py`** — streaming token-by-token responses
- **`prompts_text_usecases.py` sentiment_analysis pattern** — `INTENT_PROMPT` classifies each message as `create_posts`, `manage`, or `quit` before routing

Or run the cell below to launch directly from this notebook.

In [1]:
# The interactive demo requires a real terminal — it cannot run inside Jupyter
# because rich UI and input() calls need a proper terminal to work.
#
# To launch the demo, open a terminal in this folder and run:
#
#   python demo.py
#
# Or run the cell below to open a terminal automatically (VS Code only):

import os, subprocess, sys

demo_path = os.path.abspath('demo.py')
folder    = os.path.dirname(demo_path)

if sys.platform == 'win32':
    # Opens a new Command Prompt window and runs demo.py
    subprocess.Popen(
        f'start cmd /k python "{demo_path}"',
        shell=True, cwd=folder
    )
    print(f'✅ Launched demo.py in a new terminal window.')
elif sys.platform == 'darwin':
    # Opens Terminal.app on Mac
    subprocess.Popen(
        ['open', '-a', 'Terminal', demo_path],
        cwd=folder
    )
    print(f'✅ Launched demo.py in Terminal.')
else:
    print('Run this in your terminal:')
    print(f'  python "{demo_path}"')


✅ Launched demo.py in a new terminal window.
